# ISE-A2: Reproducible Model Walkthrough

This notebook walks through the proposed clinic appointment domain model carried forward from A1. It uses deterministic synthetic timestamps and no real patient or clinic data. The half-open arrival window `[start, end)` and 15-minute grace threshold are candidate model parameters, not clinic-approved rules. The notebook runs the test suite and records its actual output; it must fail visibly if any test fails.

In [1]:
from pathlib import Path
import json, os, platform, subprocess, sys
from datetime import datetime, timedelta, timezone

ROOT = Path.cwd()
if not (ROOT / 'src' / 'clinic_model.py').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src' / 'clinic_model.py').exists():
    raise FileNotFoundError('Run from repository root or notebook/ directory.')
sys.path.insert(0, str(ROOT))
from src.clinic_model import (
    AppointmentState, CheckInDecision, TimeWindow, decide_check_in,
)
TZ_CN = timezone(timedelta(hours=8))
def at(hour, minute=0):
    return datetime(2026, 10, 12, hour, minute, tzinfo=TZ_CN)

window = TimeWindow(at(9), at(11))
assert window.is_valid()
print('Root:', ROOT)
print('Python:', sys.version.split()[0])
print('Synthetic window:', window.start.isoformat(), 'to', window.end.isoformat())

Root: /mnt/data/ISE-A2_Submission_Package/repository
Python: 3.13.5
Synthetic window: 2026-10-12T09:00:00+08:00 to 2026-10-12T11:00:00+08:00


## 1. Time-window boundary walkthrough

The start is inclusive; the end is excluded. At the end and during the configured grace interval, a late-arrival review is required. Reaching the grace threshold makes an appointment eligible for staff no-show reconciliation, not automatically a no-show.

In [2]:
boundary_cases = [
    ('one minute before start', at(8, 59), CheckInDecision.TOO_EARLY),
    ('exactly at start', at(9, 0), CheckInDecision.ACCEPT_CHECK_IN),
    ('inside window', at(10, 15), CheckInDecision.ACCEPT_CHECK_IN),
    ('exactly at end', at(11, 0), CheckInDecision.LATE_REVIEW),
    ('inside grace', at(11, 14), CheckInDecision.LATE_REVIEW),
    ('at grace threshold', at(11, 15), CheckInDecision.NO_SHOW_ELIGIBLE),
]
boundary_results = []
for label, now, expected in boundary_cases:
    actual = decide_check_in(window, AppointmentState.CONFIRMED, now)
    assert actual == expected, (label, actual, expected)
    boundary_results.append({'case': label, 'time': now.isoformat(), 'expected': expected.value, 'actual': actual.value, 'passed': True})
boundary_results

[{'case': 'one minute before start',
  'time': '2026-10-12T08:59:00+08:00',
  'expected': 'TOO_EARLY',
  'actual': 'TOO_EARLY',
  'passed': True},
 {'case': 'exactly at start',
  'time': '2026-10-12T09:00:00+08:00',
  'expected': 'ACCEPT_CHECK_IN',
  'actual': 'ACCEPT_CHECK_IN',
  'passed': True},
 {'case': 'inside window',
  'time': '2026-10-12T10:15:00+08:00',
  'expected': 'ACCEPT_CHECK_IN',
  'actual': 'ACCEPT_CHECK_IN',
  'passed': True},
 {'case': 'exactly at end',
  'time': '2026-10-12T11:00:00+08:00',
  'expected': 'LATE_REVIEW',
  'actual': 'LATE_REVIEW',
  'passed': True},
 {'case': 'inside grace',
  'time': '2026-10-12T11:14:00+08:00',
  'expected': 'LATE_REVIEW',
  'actual': 'LATE_REVIEW',
  'passed': True},
 {'case': 'at grace threshold',
  'time': '2026-10-12T11:15:00+08:00',
  'expected': 'NO_SHOW_ELIGIBLE',
  'actual': 'NO_SHOW_ELIGIBLE',
  'passed': True}]

## 2. Execute the invariant and negative-case suite

This executes the repository test command in a separate process. The notebook stops with an error if the tests fail and stores the unmodified output for reproducibility.

In [3]:
run = subprocess.run(
    [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
    cwd=ROOT, text=True, capture_output=True
)
test_output = run.stdout + run.stderr
raw_dir = ROOT / 'raw_results'
raw_dir.mkdir(exist_ok=True)
(raw_dir / 'a2_notebook_test_output.txt').write_text(test_output, encoding='utf-8')
print(test_output)
if run.returncode != 0:
    raise RuntimeError(f'Unit test suite failed with exit code {run.returncode}; see raw_results/a2_notebook_test_output.txt')
# Extract the unittest summary, but retain full raw output above.
summary_lines = [line.strip() for line in test_output.splitlines() if line.strip().startswith('Ran ') or line.strip() == 'OK']
print('Detected summary:', summary_lines)

test_cancelled_rejected (test_clinic_model.TestCheckInDecisionTable.test_cancelled_rejected) ... ok
test_completed_is_closed (test_clinic_model.TestCheckInDecisionTable.test_completed_is_closed) ... ok
test_grace_threshold_makes_no_show_eligible_not_automatic (test_clinic_model.TestCheckInDecisionTable.test_grace_threshold_makes_no_show_eligible_not_automatic) ... ok
test_inside_window_accepted (test_clinic_model.TestCheckInDecisionTable.test_inside_window_accepted) ... ok
test_naive_current_time_raises (test_clinic_model.TestCheckInDecisionTable.test_naive_current_time_raises) ... ok
test_negative_grace_raises (test_clinic_model.TestCheckInDecisionTable.test_negative_grace_raises) ... ok
test_pre_grace_end_routes_to_late_review (test_clinic_model.TestCheckInDecisionTable.test_pre_grace_end_routes_to_late_review) ... ok
test_repeat_checkin_is_idempotent_decision (test_clinic_model.TestCheckInDecisionTable.test_repeat_checkin_is_idempotent_decision) ... ok
test_start_boundary_accepted (

In [4]:
results = {
    'artifact': 'ISE-A2 Domain Model and Testable Specification',
    'run_date': '2026-10-12',
    'data_kind': 'synthetic deterministic fixtures only',
    'python_version': sys.version.split()[0],
    'platform': platform.platform(),
    'command': 'python -m unittest discover -s tests -v',
    'exit_code': run.returncode,
    'summary_lines': summary_lines,
    'boundary_cases': boundary_results,
    'interpretation': 'Passing tests indicate conformance of this reference model to the implemented examples; they do not establish clinic policy, operational performance, legal compliance or production readiness.'
}
out = raw_dir / 'a2_notebook_results.json'
out.write_text(json.dumps(results, indent=2), encoding='utf-8')
print('Wrote', out.relative_to(ROOT))
print(json.dumps(results, indent=2))

Wrote raw_results/a2_notebook_results.json
{
  "artifact": "ISE-A2 Domain Model and Testable Specification",
  "run_date": "2026-10-12",
  "data_kind": "synthetic deterministic fixtures only",
  "python_version": "3.13.5",
  "platform": "Linux-6.18.44-x86_64-with-glibc2.41",
  "command": "python -m unittest discover -s tests -v",
  "exit_code": 0,
  "summary_lines": [
    "Ran 36 tests in 0.002s",
    "OK"
  ],
  "boundary_cases": [
    {
      "case": "one minute before start",
      "time": "2026-10-12T08:59:00+08:00",
      "expected": "TOO_EARLY",
      "actual": "TOO_EARLY",
      "passed": true
    },
    {
      "case": "exactly at start",
      "time": "2026-10-12T09:00:00+08:00",
      "expected": "ACCEPT_CHECK_IN",
      "actual": "ACCEPT_CHECK_IN",
      "passed": true
    },
    {
      "case": "inside window",
      "time": "2026-10-12T10:15:00+08:00",
      "expected": "ACCEPT_CHECK_IN",
      "actual": "ACCEPT_CHECK_IN",
      "passed": true
    },
    {
      "case": "e